In [29]:
# ============================================================
# CLARA-RR v5 — LegalSeg Adaptation
# Dataset : LegalSeg (7,120 docs, 1.4M sentences, 7 roles)
# Task    : 5-class RR labeling after None-drop + label remap
# Baseline: Hier-BiLSTM-CRF Macro-F1 = 0.77 (LegalSeg paper)
# Target  : Beat 0.77 Macro-F1 on LegalSeg test set
# Hardware: Kaggle T4 (16GB VRAM)
# ============================================================

In [30]:
# ── Cell 1: Install ─────────────────────────────────────────
!pip install transformers datasets pytorch-crf scikit-learn pandas numpy tqdm --quiet

In [31]:
# ── Cell 2: Imports ─────────────────────────────────────────
import os, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from transformers import (AutoTokenizer, AutoModel,
                          get_cosine_schedule_with_warmup)
from torchcrf import CRF
from sklearn.metrics import (classification_report,
                              f1_score, precision_score, recall_score)
from collections import Counter
from torch.optim import AdamW 
from tqdm import tqdm
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Device: cuda
GPU: Tesla T4


In [32]:
# ── Cell 3: Paths ────────────────────────────────────────────
TRAIN_CSV = "/kaggle/input/datasets/sritusharsharma/more-data/legalseg dataset/train.csv"
VAL_CSV   = "/kaggle/input/datasets/sritusharsharma/more-data/legalseg dataset/val.csv"
TEST_CSV  = "/kaggle/input/datasets/sritusharsharma/more-data/legalseg dataset/test.csv"

MODEL_NAME = "law-ai/InLegalBERT"
OUTPUT_DIR = "/kaggle/working"

In [33]:
# ── Cell 4: Config ───────────────────────────────────────────
class Config:
    # Label remapping: LegalSeg 7 → CLARA-RR 5
    # None is dropped; remaining 5 mapped to indices 0-4
    LEGALSEG_TO_CLARA = {
        "Facts"     : "Facts",
        "Reasoning" : "Ratio",
        "Decision"  : "Ruling_Present",
        "AoP"       : "Argument",
        "AoR"       : "Argument",
        "Issue"     : "Argument",   # Issue → Argument (closest legal equivalent)
        "None"      : None,         # DROP
    }

    LABEL2ID = {
        "Facts"          : 0,
        "Argument"       : 1,
        "Ratio"          : 2,
        "Ruling_Present" : 3,
    }
    ID2LABEL = {v: k for k, v in LABEL2ID.items()}
    NUM_LABELS = 4   # 4 classes after drop

    # ── Model ──────────────────────────────────────────────
    MAX_LEN         = 192   # tokens per sentence+context (T4 safe)
    CONTEXT_WINDOW  = 3     # neighbours each side (reduced from 5 for scale)
    HIDDEN_SIZE     = 768
    NUM_HEADS       = 8
    SEG_LAYERS      = 2
    DROPOUT         = 0.15

    # ── Training ───────────────────────────────────────────
    BATCH_SIZE      = 2     # documents per batch
    GRAD_ACCUM      = 16     # effective batch = 32 docs
    EPOCHS          = 10
    LR              = 2e-5
    WEIGHT_DECAY    = 0.01
    WARMUP_RATIO    = 0.1
    MAX_GRAD_NORM   = 1.0
    FREEZE_LAYERS   = 8     # freeze BERT layers 0-7 first 3 epochs

    # ── Loss weights ───────────────────────────────────────
    FOCAL_GAMMA     = 2.0
    FOCAL_WEIGHT    = 0.5
    SHIFT_WEIGHT    = 0.3
    CON_WEIGHT      = 0.3
    RULING_WEIGHT   = 0.4

    # RulingConfusionLoss: Ruling_Present only (no Ruling_Lower in LegalSeg)
    RULING_PRESENT_MULT = 4.0

    # PositionPriorBias
    POS_HIDDEN      = 32

    # LexiconEmissionBias
    LEX_FEATURES    = 12
    LEX_HIDDEN      = 48

    # SupConLoss
    CON_TEMP        = 0.3
    CON_DIM         = 128

    # Sampling: max sentences per doc (T4 memory)
    MAX_SENTS_PER_DOC = 80   # truncate very long docs during training

CFG = Config()

# Label distribution in LegalSeg train (for class weights)
# Facts:169653, Issue:12791, AoP:64987, AoR:50097,
# Reasoning:202346, Decision:19574, None:603059 (dropped)
# After remap to 5 classes:
#   Facts=169653, Argument=127875(AoP+AoR+Issue), Ratio=202346,
#   Ruling_Present=19574
# Effective counts (None dropped):
LABEL_COUNTS = {
    "Facts"          : 169653,
    "Argument"       : 127875,   # AoP + AoR + Issue merged
    "Ratio"          : 202346,
    "Ruling_Present" : 19574,
}

print("Config loaded.")
print(f"Classes: {CFG.LABEL2ID}")
print(f"Label counts after remap: {LABEL_COUNTS}")

Config loaded.
Classes: {'Facts': 0, 'Argument': 1, 'Ratio': 2, 'Ruling_Present': 3}
Label counts after remap: {'Facts': 169653, 'Argument': 127875, 'Ratio': 202346, 'Ruling_Present': 19574}


In [34]:
# ── Cell 5: Compute class weights ───────────────────────────
total = sum(LABEL_COUNTS.values())
# Effective number weighting (Cui et al. 2019)
beta = 0.9999
eff_num = {k: (1 - beta**v) / (1 - beta) for k, v in LABEL_COUNTS.items()}
weights = {k: 1.0 / eff_num[k] for k in eff_num}
wsum = sum(weights.values())
weights = {k: v / wsum * CFG.NUM_LABELS for k, v in weights.items()}

CLASS_WEIGHTS = torch.tensor(
    [weights[CFG.ID2LABEL[i]] for i in range(CFG.NUM_LABELS)],
    dtype=torch.float32
).to(DEVICE)
print(f"Class weights: {CLASS_WEIGHTS}")

Class weights: tensor([0.9605, 0.9605, 0.9605, 1.1185], device='cuda:0')


In [35]:
# ── Cell 6: Load & preprocess CSVs ──────────────────────────
def load_and_remap(csv_path, split_name="train"):
    df = pd.read_csv(
        csv_path,
        encoding="utf-8-sig",
        engine="python",
        on_bad_lines="skip"
    )
    print(f"\n[{split_name}] Loaded {len(df):,} rows")

    # Clean column names
    df.columns = (df.columns.str.strip()
                             .str.replace("\ufeff","",regex=False)
                             .str.lower())

    # Rename to pipeline format
    df = df.rename(columns={
        "text"  : "sentence",
        "label" : "raw_label",
        "index" : "doc_id",      # Index IS the document ID
    })

    df["sentence"]  = df["sentence"].astype(str).str.strip()
    df["raw_label"] = df["raw_label"].astype(str).str.strip()
    df = df[df["sentence"] != ""].reset_index(drop=True)

    print(f"  Unique docs: {df['doc_id'].nunique():,}")
    print(f"  Raw labels: {df['raw_label'].value_counts().to_dict()}")

    # ── Apply CLARA label remapping from Config ──────────────
    df["label"] = df["raw_label"].map(CFG.LEGALSEG_TO_CLARA)

    before = len(df)
    df = df[df["label"].notna()].copy()
    after  = len(df)
    print(f"  After None-drop: {after:,}/{before:,} ({100*after/before:.1f}%)")

    df["label_id"] = df["label"].map(CFG.LABEL2ID)
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)

    print(f"  Final label dist: {df['label'].value_counts().to_dict()}")
    print(f"  Unique docs after drop: {df['doc_id'].nunique():,}")
    return df

In [36]:
# ── Cell 7: Build document list ──────────────────────────────
def build_doc_list(df, max_sents=None):
    """Group sentences by doc_id into list of dicts."""
    docs = []
    for doc_id, grp in df.groupby("doc_id", sort=False):
        sents = grp["sentence"].fillna("").tolist()
        labels = grp["label_id"].tolist()
        if max_sents and len(sents) > max_sents:
            sents  = sents[:max_sents]
            labels = labels[:max_sents]
        if len(sents) < 2:
            continue
        docs.append({
            "doc_id"   : doc_id,
            "sentences": sents,
            "labels"   : labels,
            "n_sents"  : len(sents),
        })
    return docs

In [37]:
# ── Cell 8: Tokenizer & Dataset ──────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class LegalSegDataset(Dataset):
    def __init__(self, docs, tokenizer, max_len=256, context_window=3,
                 max_sents=150, augment=False):
        self.docs    = docs
        self.tok     = tokenizer
        self.max_len = max_len
        self.ctx     = context_window
        self.max_s   = max_sents
        self.augment = augment

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        doc = self.docs[idx]
        sents  = doc["sentences"]
        labels = doc["labels"]
        n      = len(sents)

        # Encode each sentence with context window
        input_ids_list      = []
        attention_mask_list = []

        for i in range(n):
            lo = max(0, i - self.ctx)
            hi = min(n, i + self.ctx + 1)
            ctx_sents = sents[lo:hi]
            text = " [SEP] ".join(ctx_sents)

            enc = self.tok(
                text,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            input_ids_list.append(enc["input_ids"].squeeze(0))
            attention_mask_list.append(enc["attention_mask"].squeeze(0))

        input_ids      = torch.stack(input_ids_list)       # [n, max_len]
        attention_masks= torch.stack(attention_mask_list)  # [n, max_len]
        labels_t       = torch.tensor(labels, dtype=torch.long)

        # Relative positions [0.0, ..., 1.0]
        if n > 1:
            positions = torch.linspace(0.0, 1.0, n)
        else:
            positions = torch.tensor([0.5])

        # Lexicon features [n, LEX_FEATURES]
        lex = self._lex_features(sents)

        return {
            "input_ids"      : input_ids,
            "attention_masks": attention_masks,
            "labels"         : labels_t,
            "positions"      : positions,
            "lex"            : lex,
            "n_sents"        : n,
        }

    def _lex_features(self, sents):
        """12 handcrafted legal features per sentence."""
        # Role-specific keywords (5-class adapted)
        kw = {
            0: ["facts","factual","background","event","incident",
                "appellant","respondent","plaintiff","defendant"],
            1: ["submitted","argued","contended","counsel","petitioner",
                "respondent","claimed","issue","question"],
            2: ["therefore","thus","accordingly","ratio","reasoning",
                "principle","law","held","observed","applying"],
            3: ["disposed","dismissed","allowed","order","judgment",
                "direct","hereby","accordingly","relief"],
        }
        feats = []
        for s in sents:
            sl = s.lower()
            words = sl.split()
            nw = max(len(words), 1)
            row = []
            # 4 keyword density features (one per class)
            for role_id in range(CFG.NUM_LABELS):
                cnt = sum(1 for w in kw[role_id] if w in sl)
                row.append(cnt / nw)
            # 5 structural features
            row.append(min(nw / 50.0, 1.0))                      # length
            row.append(sl.count('"') / max(nw, 1))               # quote density
            row.append(sum(c.isdigit() for c in s) / max(len(s), 1))  # digit density
            row.append(1.0 if "?" in s else 0.0)                 # question
            row.append(1.0 if " section " in sl or "sec." in sl else 0.0)  # statute ref
            # 3 citation signals
            row.append(1.0 if " v." in sl or " vs " in sl else 0.0)    # case citation
            row.append(1.0 if "supra" in sl or "ibid" in sl else 0.0)  # precedent ref
            row.append(1.0 if "article" in sl or "act" in sl else 0.0) # statute
            feats.append(row)
        return torch.tensor(feats, dtype=torch.float32)  # [n, 12]

def collate_fn(batch):
    """Pad variable-length documents to same n_sents."""
    max_n = max(b["n_sents"] for b in batch)
    max_len = batch[0]["input_ids"].shape[1]
    B = len(batch)

    input_ids       = torch.zeros(B, max_n, max_len, dtype=torch.long)
    attention_masks = torch.zeros(B, max_n, max_len, dtype=torch.long)
    labels          = torch.full((B, max_n), -1, dtype=torch.long)
    positions       = torch.zeros(B, max_n)
    lex             = torch.zeros(B, max_n, CFG.LEX_FEATURES)
    sent_mask       = torch.zeros(B, max_n, dtype=torch.bool)

    for i, b in enumerate(batch):
        n = b["n_sents"]
        input_ids[i, :n]       = b["input_ids"]
        attention_masks[i, :n] = b["attention_masks"]
        labels[i, :n]          = b["labels"]
        positions[i, :n]       = b["positions"]
        lex[i, :n]             = b["lex"]
        sent_mask[i, :n]       = True

    return {
        "input_ids"      : input_ids,
        "attention_masks": attention_masks,
        "labels"         : labels,
        "positions"      : positions,
        "lex"            : lex,
        "sent_mask"      : sent_mask,
    }

Loading tokenizer...


In [38]:
# ── Cell 9: Model Components ─────────────────────────────────

class PositionPriorBias(nn.Module):
    """MLP: relative_pos (scalar) → per-class emission bias."""
    def __init__(self, hidden=32, num_labels=4):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, hidden), nn.GELU(),
            nn.Linear(hidden, num_labels)
        )
    def forward(self, positions):
        # positions: [B, n] → [B, n, num_labels]
        return self.mlp(positions.unsqueeze(-1))

class LexiconEmissionBias(nn.Module):
    """MLP: lex_features → per-class emission bias."""
    def __init__(self, in_dim=12, hidden=48, num_labels=4):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, num_labels)
        )
    def forward(self, lex):
        return self.mlp(lex)  # [B, n, num_labels]

class LearnedEmissionScaler(nn.Module):
    """Per-class learnable temperature."""
    def __init__(self, num_labels=4):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(num_labels))
    def forward(self, emissions):
        return emissions * self.scale

class RoleAwareAttention(nn.Module):
    """Multi-head self-attention over sentences in a document."""
    def __init__(self, hidden=768, num_heads=8, dropout=0.1, num_labels=4):
        super().__init__()
        self.attn    = nn.MultiheadAttention(hidden, num_heads,
                                              dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(dropout)
        # For role-conditioned second pass
        self.role_emb = nn.Embedding(num_labels + 1, hidden, padding_idx=num_labels)

    def forward(self, x, mask=None, role_ids=None):
        # x: [B, n, H], mask: [B, n] (True = valid)
        if role_ids is not None:
            x = x + self.role_emb(role_ids)
        key_padding_mask = ~mask if mask is not None else None
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        return self.norm(x + self.dropout(attn_out))

class SegmentTransformer(nn.Module):
    """2-layer Pre-LN transformer for segment-level patterns."""
    def __init__(self, hidden=768, num_heads=8, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=num_heads, dropout=dropout,
            batch_first=True, norm_first=True,  # Pre-LN
            dim_feedforward=hidden * 2
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.norm    = nn.LayerNorm(hidden)

    def forward(self, x, mask=None):
        src_key_padding_mask = ~mask if mask is not None else None
        out = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        return self.norm(out)

class PseudoLabelRefiner(nn.Module):
    """Two-pass: first decode → pseudo labels → role-aware re-encode."""
    def __init__(self, hidden=768, num_labels=4):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.Sigmoid()
        )
        self.role_attn = RoleAwareAttention(hidden, num_labels=num_labels)

    def forward(self, doc_repr, emissions, sent_mask, role_attn_module):
        # Greedy pseudo-labels from emissions
        pseudo_labels = emissions.argmax(dim=-1)  # [B, n]
        # Mask padding positions
        pseudo_labels = pseudo_labels.masked_fill(~sent_mask, CFG.NUM_LABELS)

        # Second pass with role conditioning
        refined = role_attn_module(doc_repr, mask=sent_mask,
                                   role_ids=pseudo_labels)
        gate_input = torch.cat([doc_repr, refined], dim=-1)
        gate = self.gate(gate_input)
        return doc_repr + gate * (refined - doc_repr)

class ShiftHead(nn.Module):
    """Binary classifier for role boundary detection."""
    def __init__(self, hidden=768):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(hidden, 1)        )
    def forward(self, x):
        return self.head(x).squeeze(-1)  # [B, n]

In [39]:
# ── Cell 10: Full CLARA-RR Model ─────────────────────────────
class CLARAModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # BERT encoder
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        H = cfg.HIDDEN_SIZE
        L = cfg.NUM_LABELS
        # Positional encoding
        self.abs_pos_emb = nn.Embedding(2048, H)
        self.rel_pos_proj= nn.Sequential(nn.Linear(1, H), nn.GELU())
        self.end_pos_emb = nn.Embedding(2048, H)
        self.pos_gate = nn.Sequential(nn.Linear(H, H), nn.Sigmoid())
        self.pos_norm = nn.LayerNorm(H)
        # Document-level modules
        self.role_attn = RoleAwareAttention(H, cfg.NUM_HEADS, cfg.DROPOUT, L)
        self.seg_transform = SegmentTransformer(H, cfg.NUM_HEADS, cfg.DROPOUT)
        self.dropout = nn.Dropout(cfg.DROPOUT)
        # Emission signals
        self.classifier = nn.Sequential(
            nn.Linear(H, H // 2), nn.GELU(), nn.LayerNorm(H // 2),
            nn.Linear(H // 2, L)
        )
        self.lex_bias = LexiconEmissionBias(cfg.LEX_FEATURES, cfg.LEX_HIDDEN, L)
        self.pos_bias = PositionPriorBias(cfg.POS_HIDDEN, L)
        self.scaler = LearnedEmissionScaler(L)
        # Pseudo-label refiner
        self.refiner = PseudoLabelRefiner(H, L)
        # Shift detection head
        self.shift_head = ShiftHead(H)
        # Contrastive projection
        self.con_proj = nn.Sequential(
            nn.Linear(H, cfg.CON_DIM), nn.GELU(),
            nn.Linear(cfg.CON_DIM, cfg.CON_DIM)
        )
        # CRF
        self.crf = CRF(L, batch_first=True)
        # CRF structural prior: Ruling_Present (3) is terminal
        with torch.no_grad():
            rp = cfg.LABEL2ID["Ruling_Present"]
            self.crf.transitions[0, rp] -= 1.5
            self.crf.transitions[1, rp] -= 1.5
            self.crf.transitions[rp, 0] -= 1.5
            self.crf.transitions[rp, 1] -= 1.5
        self._cache = {}

    def _encode_bert(self, input_ids, attention_masks):
        """Encode [B, n, max_len] → [B, n, H]."""
        B, n, L = input_ids.shape
        flat_ids = input_ids.view(B * n, L)
        flat_masks = attention_masks.view(B * n, L)
        # Chunk to avoid OOM on large docs
        chunk_size = 32
        reps = []
        for start in range(0, B * n, chunk_size):
            end = min(start + chunk_size, B * n)
            out = self.bert(
                input_ids=flat_ids[start:end],
                attention_mask=flat_masks[start:end]
            )
            # Mean pooling
            mask_exp = flat_masks[start:end].unsqueeze(-1).float()
            rep = (out.last_hidden_state * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1)
            reps.append(rep)
        reps = torch.cat(reps, dim=0)
        return reps.view(B, n, -1)

    def _add_position(self, h, n, B):
        """Fuse three position signals into h."""
        idx = torch.arange(n, device=h.device)
        abs_e = self.abs_pos_emb(idx.clamp(max=2047)).unsqueeze(0).expand(B, -1, -1)
        rel_e = self.rel_pos_proj(
            torch.linspace(0, 1, n, device=h.device).unsqueeze(0).unsqueeze(-1).expand(B, -1, -1)
        )
        end_idx = torch.arange(n - 1, -1, -1, device=h.device)
        end_e = self.end_pos_emb(end_idx.clamp(max=2047)).unsqueeze(0).expand(B, -1, -1)
        gate = self.pos_gate(h)
        fused = h + 0.5 * gate * (abs_e + rel_e + end_e)
        return self.pos_norm(fused)

    def _apply_biases(self, doc_repr, positions, lex):
        e_cls = self.classifier(doc_repr)
        e_lex = self.lex_bias(lex)
        e_pos = self.pos_bias(positions)
        return self.scaler(e_cls + e_lex + e_pos)

    def forward(self, input_ids, attention_masks, labels,
                positions, lex, sent_mask):
        B, n, _ = input_ids.shape
        # 1. BERT encode
        h = self._encode_bert(input_ids, attention_masks)
        # 2. Position encoding
        h = self._add_position(h, n, B)
        # 3. Document-level attention
        h = self.role_attn(h, mask=sent_mask)
        h = self.seg_transform(h, mask=sent_mask)
        h = self.dropout(h)
        # 4. First-pass emissions
        emissions1 = self._apply_biases(h, positions, lex)
        # 5. Pseudo-label refinement
        h = self.refiner(h, emissions1, sent_mask, self.role_attn)
        # 6. Final emissions
        emissions = self._apply_biases(h, positions, lex)

        # 7. CRF loss
        valid_labels = labels.clone()
        valid_labels[~sent_mask] = 0
        crf_mask = sent_mask.bool()
        crf_loss = -self.crf(emissions, valid_labels, mask=crf_mask, reduction="mean")

        # 8. Focal loss
        flat_e = emissions[sent_mask]
        flat_l = labels[sent_mask]
        ce = F.cross_entropy(flat_e, flat_l, weight=CLASS_WEIGHTS, reduction="none")
        pt = torch.exp(-ce)
        focal_loss = ((1 - pt) ** self.cfg.FOCAL_GAMMA * ce).mean()

        # 9. Shift loss
        shift_logits = self.shift_head(h).squeeze(-1)
        shift_true = torch.zeros_like(shift_logits)
        lbl_padded = labels.clone().float()
        lbl_padded[~sent_mask] = -1
        shift_true[:, 1:] = (lbl_padded[:, 1:] != lbl_padded[:, :-1]).float()
        shift_true[:, 0] = 0.0

        shift_mask = sent_mask.float()
        pos_w = torch.where(shift_true == 1, 3.0, 1.0)
        shift_loss = F.binary_cross_entropy_with_logits(
            shift_logits, shift_true, reduction="none"
        )
        shift_loss = (shift_loss * pos_w * shift_mask).sum() / shift_mask.sum().clamp(min=1)

        # 10. Supervised contrastive loss — FIXED (fp16 safe)
        flat_h = h[sent_mask]
        flat_l = labels[sent_mask]

        # Force fp32 for SupCon to prevent overflow in autocast
        with torch.cuda.amp.autocast(enabled=False):
            flat_h = flat_h.float()
            proj = F.normalize(self.con_proj(flat_h), dim=-1)
            con_loss = self._supcon_loss(proj, flat_l)

        # 11. RulingConfusionLoss
        rp_id = self.cfg.LABEL2ID["Ruling_Present"]
        rp_mask = (flat_l == rp_id)
        ruling_loss = torch.tensor(0.0, device=DEVICE)
        if rp_mask.any():
            rp_ce = F.cross_entropy(
                flat_e[rp_mask], flat_l[rp_mask], weight=CLASS_WEIGHTS
            )
            ruling_loss = self.cfg.RULING_PRESENT_MULT * rp_ce

        # 12. Total loss
        total = (crf_loss
                 + self.cfg.FOCAL_WEIGHT * focal_loss
                 + self.cfg.SHIFT_WEIGHT * shift_loss
                 + self.cfg.CON_WEIGHT * con_loss
                 + self.cfg.RULING_WEIGHT * ruling_loss)
        return total, emissions

    def _supcon_loss(self, proj, labels, temp=0.3):
        """Safe Supervised Contrastive Loss"""
        if proj.shape[0] < 2:
            return torch.tensor(0.0, device=proj.device)

        # Safe diagonal mask for fp16 / fp32
        mask_value = -1e4 if proj.dtype == torch.float16 else -1e9

        sim = torch.mm(proj, proj.T) / temp
        sim.fill_diagonal_(mask_value)

        mask = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        mask.fill_diagonal_(0)

        if mask.sum() == 0:
            return torch.tensor(0.0, device=proj.device)

        log_prob = F.log_softmax(sim, dim=1)
        loss = -(mask * log_prob).sum(1) / mask.sum(1).clamp(min=1)
        return loss.mean()

    @torch.no_grad()
    def predict(self, input_ids, attention_masks, positions, lex, sent_mask):
        B, n, _ = input_ids.shape
        h = self._encode_bert(input_ids, attention_masks)
        h = self._add_position(h, n, B)
        h = self.role_attn(h, mask=sent_mask)
        h = self.seg_transform(h, mask=sent_mask)
        emissions1 = self._apply_biases(h, positions, lex)
        h = self.refiner(h, emissions1, sent_mask, self.role_attn)
        emissions = self._apply_biases(h, positions, lex)
        preds = self.crf.decode(emissions, mask=sent_mask.bool())
        return preds

In [40]:
# ── Cell 11: Freeze/Unfreeze helpers ────────────────────────
def freeze_bert_layers(model, n_layers):
    for name, param in model.bert.named_parameters():
        if "embeddings" in name:
            param.requires_grad = False
        elif "encoder.layer" in name:
            layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
            param.requires_grad = (layer_num >= n_layers)

def unfreeze_all(model, lr_factor=0.5, optimizer=None):
    for param in model.bert.parameters():
        param.requires_grad = True
    if optimizer is not None:
        for g in optimizer.param_groups:
            if g.get("is_bert", False):
                g["lr"] = g["lr"] * lr_factor

In [41]:
# ── Cell 12: Training loop ───────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss = 0
    n_batches = 0
    optimizer.zero_grad()

    # Progressive unfreeze at epoch 4
    if epoch == 4:
        print("  → Unfreezing all BERT layers at 0.5× LR")
        unfreeze_all(model, optimizer=optimizer)

    for step, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch}")):
        input_ids       = batch["input_ids"].to(DEVICE)
        attention_masks = batch["attention_masks"].to(DEVICE)
        labels          = batch["labels"].to(DEVICE)
        positions       = batch["positions"].to(DEVICE)
        lex             = batch["lex"].to(DEVICE)
        sent_mask       = batch["sent_mask"].to(DEVICE)

        with autocast():
            loss, _ = model(input_ids, attention_masks, labels,
                            positions, lex, sent_mask)
            loss = loss / CFG.GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % CFG.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG.GRAD_ACCUM
        n_batches  += 1

        if step % 200 == 0:
            print(f"    Step {step} | Loss {total_loss/n_batches:.4f}")

    return total_loss / n_batches

@torch.no_grad()
def evaluate(model, loader, split="val"):
    model.eval()
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc=f"Eval [{split}]"):
        input_ids       = batch["input_ids"].to(DEVICE)
        attention_masks = batch["attention_masks"].to(DEVICE)
        positions       = batch["positions"].to(DEVICE)
        lex             = batch["lex"].to(DEVICE)
        sent_mask       = batch["sent_mask"].to(DEVICE)
        labels          = batch["labels"]

        preds = model.predict(input_ids, attention_masks,
                              positions, lex, sent_mask)

        for i, pred_seq in enumerate(preds):
            true_seq = labels[i][sent_mask[i].cpu()].tolist()
            all_preds.extend(pred_seq)
            all_labels.extend(true_seq)

    macro_f1 = f1_score(all_labels, all_preds, average="macro",
                         labels=list(range(CFG.NUM_LABELS)), zero_division=0)
    report = classification_report(
        all_labels, all_preds,
        labels=list(range(CFG.NUM_LABELS)),
        target_names=[CFG.ID2LABEL[i] for i in range(CFG.NUM_LABELS)],
        zero_division=0
    )
    print(f"\n[{split}] Macro-F1: {macro_f1:.4f}")
    print(report)
    return macro_f1

In [ ]:
# ── Cell 13: Main training ───────────────────────────────────
print("Loading CSVs...")
train_df = load_and_remap(TRAIN_CSV, "train")
val_df   = load_and_remap(VAL_CSV,   "val")
test_df  = load_and_remap(TEST_CSV,  "test")

print("\nBuilding document lists...")
train_docs = build_doc_list(train_df, max_sents=CFG.MAX_SENTS_PER_DOC)
val_docs   = build_doc_list(val_df,   max_sents=None)
test_docs  = build_doc_list(test_df,  max_sents=None)

print(f"Train docs: {len(train_docs)}")
print(f"Val docs  : {len(val_docs)}")
print(f"Test docs : {len(test_docs)}")

# Datasets
train_ds = LegalSegDataset(train_docs, tokenizer, CFG.MAX_LEN,
                            CFG.CONTEXT_WINDOW, CFG.MAX_SENTS_PER_DOC)
val_ds   = LegalSegDataset(val_docs,   tokenizer, CFG.MAX_LEN,
                            CFG.CONTEXT_WINDOW)
test_ds  = LegalSegDataset(test_docs,  tokenizer, CFG.MAX_LEN,
                            CFG.CONTEXT_WINDOW)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=2, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)

# Model
print("\nBuilding model...")
model = CLARAModel(CFG).to(DEVICE)

# Freeze lower BERT layers initially
freeze_bert_layers(model, CFG.FREEZE_LAYERS)

# Optimizer — separate LR for BERT vs heads
bert_params = list(model.bert.parameters())
head_params = [p for n, p in model.named_parameters()
               if not n.startswith("bert.") and p.requires_grad]

optimizer = AdamW([
    {"params": bert_params, "lr": CFG.LR,       "is_bert": True},
    {"params": head_params, "lr": CFG.LR * 5,   "is_bert": False},
], weight_decay=CFG.WEIGHT_DECAY)

total_steps = (len(train_loader) // CFG.GRAD_ACCUM) * CFG.EPOCHS
warmup_steps = int(total_steps * CFG.WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = GradScaler()

print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")
print(f"Train batches per epoch: {len(train_loader)}")

best_val_f1 = 0.0
best_epoch  = 0
results = []

for epoch in range(1, CFG.EPOCHS + 1):
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch}/{CFG.EPOCHS}")
    print(f"{'='*60}")

    train_loss = train_epoch(model, train_loader, optimizer,
                              scheduler, scaler, epoch)
    print(f"Train Loss: {train_loss:.4f}")

    val_f1 = evaluate(model, val_loader, split="val")
    results.append({"epoch": epoch, "train_loss": train_loss, "val_f1": val_f1})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        torch.save(model.state_dict(),
                   f"{OUTPUT_DIR}/clara_legalseg_best.pt")
        print(f"  ✓ New best! Saved.")

    torch.save(model.state_dict(),
               f"{OUTPUT_DIR}/clara_legalseg_epoch{epoch}.pt")

print(f"\nBest Val Macro-F1: {best_val_f1:.4f} at Epoch {best_epoch}")

Loading CSVs...

[train] Loaded 1,122,507 rows
  Unique docs: 4,984
  Raw labels: {'nan': 602988, 'Reasoning': 202328, 'Facts': 169647, 'Arguments of Petitioner': 64985, 'Arguments of Respondent': 50088, 'Decision': 19571, 'Issue': 12769}
  After None-drop: 404,315/1,122,376 (36.0%)
  Final label dist: {'Ratio': 202328, 'Facts': 169647, 'Ruling_Present': 19571, 'Argument': 12769}
  Unique docs after drop: 4,984

[val] Loaded 230,264 rows
  Unique docs: 1,134
  Raw labels: {'nan': 102266, 'Reasoning': 47516, 'Facts': 41515, 'Arguments of Petitioner': 18069, 'Arguments of Respondent': 11853, 'Decision': 5697, 'Issue': 3340}
  After None-drop: 98,068/230,256 (42.6%)
  Final label dist: {'Ratio': 47516, 'Facts': 41515, 'Ruling_Present': 5697, 'Argument': 3340}
  Unique docs after drop: 1,134

[test] Loaded 149,881 rows
  Unique docs: 712
  Raw labels: {'nan': 58496, 'Reasoning': 36688, 'Facts': 24908, 'Arguments of Petitioner': 14520, 'Arguments of Respondent': 9579, 'Decision': 3841, 'Iss

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total steps: 1550 | Warmup: 155
Train batches per epoch: 2492

EPOCH 1/10


Epoch 1:   0%|          | 1/2492 [00:00<41:22,  1.00it/s]

...

Epoch 5:  64%|██████▍   | 1601/2492 [29:21<15:54,  1.07s/it]

    Step 1600 | Loss 13.9911


Epoch 5:  72%|███████▏  | 1801/2492 [33:01<13:25,  1.17s/it]

    Step 1800 | Loss 14.0959


Epoch 5:  78%|███████▊  | 1936/2492 [35:26<07:26,  1.25it/s]

In [ ]:
# ── Cell 14: Final test evaluation ──────────────────────────
print("\nLoading best model for test evaluation...")
model.load_state_dict(
    torch.load(f"{OUTPUT_DIR}/clara_legalseg_best.pt", map_location=DEVICE)
)
test_f1 = evaluate(model, test_loader, split="test")

In [ ]:
# ── Cell 15: Save results summary ───────────────────────────
summary = {
    "dataset"         : "LegalSeg",
    "model"           : "CLARA-RR v5",
    "num_classes"     : CFG.NUM_LABELS,
    "classes"         : list(CFG.LABEL2ID.keys()),
    "label_remap"     : CFG.LEGALSEG_TO_CLARA,
    "best_val_macro_f1" : best_val_f1,
    "test_macro_f1"   : test_f1,
    "best_epoch"      : best_epoch,
    "epoch_results"   : results,
    "baseline"        : {
        "model"    : "Hier-BiLSTM-CRF (LegalSeg paper)",
        "macro_f1" : 0.77,
    }
}

with open(f"{OUTPUT_DIR}/clara_legalseg_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(f"Dataset        : LegalSeg (7,120 docs, 4 classes after remap)")
print(f"Best Val F1    : {best_val_f1:.4f}")
print(f"Test Macro-F1  : {test_f1:.4f}")
print(f"Baseline (BiLSTM-CRF): 0.7700")
print(f"Delta vs baseline: {test_f1 - 0.77:+.4f}")
print("="*60)

In [ ]:
# ── Cell 16: Per-class analysis ──────────────────────────────
print("\nNote on paper reporting:")
print("CLARA-RR on LegalSeg is a 4-class task (Facts, Argument, Ratio, Ruling_Present).")
print("Ruling_Lower, Precedent, Statute are absent from LegalSeg annotation schema.")
print("These results are reported as a SEPARATE experiment from Malik 7-class results.")
print("The two experiments are complementary:")
print("  - Malik 7-class: Full schema, 100 docs, Macro-F1=0.6254")
print("  - LegalSeg 4-class: Reduced schema, 7120 docs, Macro-F1=TBD")

In [ ]:
# ── Cell 17: Visualisation Setup ────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import confusion_matrix
import seaborn as sns
import itertools
 
# ── Shared style ────────────────────────────────────────────
COLORS = {
    "Facts"          : "#4472C4",
    "Argument"       : "#ED7D31",
    "Ratio"          : "#A9D18E",
    "Ruling_Present" : "#FF0000",
    "baseline"       : "#7F7F7F",
    "clara"          : "#4472C4",
}
CLASS_NAMES = [CFG.ID2LABEL[i] for i in range(CFG.NUM_LABELS)]
SAVE = lambda name: plt.savefig(f"{OUTPUT_DIR}/{name}", dpi=150,
                                 bbox_inches="tight")
 
plt.rcParams.update({
    "font.family"    : "DejaVu Sans",
    "font.size"      : 11,
    "axes.titlesize" : 13,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})
 
print("Visualisation setup complete.")

In [ ]:
# ── Cell 18: Dataset Statistics Plot ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("LegalSeg Dataset — Label Distribution After Remapping",
             fontsize=14, fontweight="bold", y=1.02)
 
# --- Before remap (LegalSeg original) ---
raw_labels  = ["Facts", "Reasoning", "AoP", "AoR", "Issue", "Decision", "None"]
raw_counts  = [169653+51924+24909,
               202346+67113+36689,
               64987+24707+14520,
               50097+16021+9579,
               12791+4259+1843,
               19574+7634+3841,
               603059+121712+58500]
raw_colors  = ["#4472C4","#A9D18E","#ED7D31","#FFC000","#5B9BD5","#FF0000","#BFBFBF"]
 
ax = axes[0]
bars = ax.barh(raw_labels, raw_counts, color=raw_colors, edgecolor="white")
ax.set_xlabel("Total Sentences (train+val+test)")
ax.set_title("Before Remapping (7 LegalSeg Labels)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
for bar, cnt in zip(bars, raw_counts):
    ax.text(bar.get_width() + 5000, bar.get_y() + bar.get_height()/2,
            f"{cnt:,}", va="center", fontsize=9)
ax.axvline(x=603059+121712+58500, color="red", linestyle="--",
           alpha=0.3, label="None class size")
 
# --- After remap (CLARA-RR 4-class) ---
clara_labels = ["Facts", "Argument\n(AoP+AoR+Issue)", "Ratio\n(Reasoning)",
                "Ruling_Present\n(Decision)"]
clara_counts = [169653+51924+24909,
                (64987+24707+14520)+(50097+16021+9579)+(12791+4259+1843),
                202346+67113+36689,
                19574+7634+3841]
clara_colors = [COLORS[k] for k in ["Facts","Argument","Ratio","Ruling_Present"]]
 
ax2 = axes[1]
bars2 = ax2.barh(clara_labels, clara_counts, color=clara_colors, edgecolor="white")
ax2.set_xlabel("Total Sentences (train+val+test)")
ax2.set_title("After Remapping (4 CLARA-RR Labels)")
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
for bar, cnt in zip(bars2, clara_counts):
    ax2.text(bar.get_width() + 3000, bar.get_y() + bar.get_height()/2,
             f"{cnt:,}", va="center", fontsize=9)
 
plt.tight_layout()
SAVE("viz_01_dataset_distribution.png")
plt.show()
print("Saved: viz_01_dataset_distribution.png")

In [ ]:
# ── Cell 19: Training Curves ─────────────────────────────────
epochs_list  = [r["epoch"]      for r in results]
losses       = [r["train_loss"] for r in results]
val_f1s      = [r["val_f1"]     for r in results]
 
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("CLARA-RR v5 — LegalSeg Training Progress", fontsize=14, fontweight="bold")
 
# Loss curve
ax = axes[0]
ax.plot(epochs_list, losses, "o-", color=COLORS["clara"],
        linewidth=2, markersize=7, label="Train Loss")
ax.axvline(x=best_epoch, color="red", linestyle="--", alpha=0.6,
           label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training Loss"); ax.legend()
ax.set_xticks(epochs_list)
 
# Val F1 curve
ax2 = axes[1]
ax2.plot(epochs_list, val_f1s, "s-", color="#ED7D31",
         linewidth=2, markersize=7, label="Val Macro-F1")
ax2.axhline(y=0.77, color=COLORS["baseline"], linestyle="--",
            linewidth=1.5, label="BiLSTM-CRF baseline (0.77)")
ax2.axhline(y=best_val_f1, color="green", linestyle=":",
            linewidth=1.5, label=f"Best val F1 ({best_val_f1:.4f})")
ax2.scatter([best_epoch], [best_val_f1], color="green",
            zorder=5, s=100, marker="*")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Macro-F1")
ax2.set_title("Validation Macro-F1"); ax2.legend()
ax2.set_xticks(epochs_list)
ax2.set_ylim(0, 1.0)
 
plt.tight_layout()
SAVE("viz_02_training_curves.png")
plt.show()
print("Saved: viz_02_training_curves.png")

In [ ]:
# ── Cell 20: Confusion Matrix (Test Set) ────────────────────
# Collect all test predictions and true labels
model.eval()
all_preds_cm, all_labels_cm = [], []
 
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Collecting test preds"):
        input_ids       = batch["input_ids"].to(DEVICE)
        attention_masks = batch["attention_masks"].to(DEVICE)
        positions       = batch["positions"].to(DEVICE)
        lex             = batch["lex"].to(DEVICE)
        sent_mask       = batch["sent_mask"].to(DEVICE)
        labels          = batch["labels"]
 
        preds = model.predict(input_ids, attention_masks,
                              positions, lex, sent_mask)
        for i, pred_seq in enumerate(preds):
            true_seq = labels[i][sent_mask[i].cpu()].tolist()
            all_preds_cm.extend(pred_seq)
            all_labels_cm.extend(true_seq)
 
cm = confusion_matrix(all_labels_cm, all_preds_cm,
                       labels=list(range(CFG.NUM_LABELS)))
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CLARA-RR v5 on LegalSeg — Test Set Confusion Matrix",
             fontsize=14, fontweight="bold")
 
# Raw counts
for ax_idx, (data, title, fmt) in enumerate([
    (cm,                    "Counts",       "d"),
    (cm.astype("float") / cm.sum(axis=1, keepdims=True).clip(min=1),
                            "Row-Normalised (Recall)", ".2f"),
]):
    ax = axes[ax_idx]
    sns.heatmap(data, annot=True, fmt=fmt, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cmap="Blues" if ax_idx == 0 else "RdYlGn",
                linewidths=0.5, linecolor="white",
                cbar_kws={"shrink": 0.8})
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
 
plt.tight_layout()
SAVE("viz_03_confusion_matrix.png")
plt.show()
print("Saved: viz_03_confusion_matrix.png")

In [ ]:
# ── Cell 21: Per-class P/R/F1 Bar Chart ─────────────────────
from sklearn.metrics import precision_recall_fscore_support
 
prec, rec, f1, support = precision_recall_fscore_support(
    all_labels_cm, all_preds_cm,
    labels=list(range(CFG.NUM_LABELS)),
    zero_division=0
)
 
fig, ax = plt.subplots(figsize=(11, 5))
x      = np.arange(CFG.NUM_LABELS)
width  = 0.25
 
b1 = ax.bar(x - width, prec, width, label="Precision",
            color="#4472C4", edgecolor="white", linewidth=0.5)
b2 = ax.bar(x,          rec,  width, label="Recall",
            color="#ED7D31", edgecolor="white", linewidth=0.5)
b3 = ax.bar(x + width,  f1,   width, label="F1-Score",
            color="#A9D18E", edgecolor="white", linewidth=0.5)
 
# Value labels
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8)
 
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=20, ha="right")
ax.set_ylabel("Score"); ax.set_ylim(0, 1.1)
ax.set_title("CLARA-RR v5 — Per-Class Precision / Recall / F1 (LegalSeg Test Set)",
             fontweight="bold")
ax.legend(loc="upper right")
ax.axhline(y=test_f1, color="red", linestyle="--", alpha=0.5,
           label=f"Macro-F1 = {test_f1:.4f}")
ax.legend(loc="upper right")
 
# Add support counts as secondary x-axis annotations
for i, sup in enumerate(support):
    ax.text(x[i], -0.08, f"n={sup:,}", ha="center",
            fontsize=8, color="gray", transform=ax.get_xaxis_transform())
 
plt.tight_layout()
SAVE("viz_04_perclass_metrics.png")
plt.show()
print("Saved: viz_04_perclass_metrics.png")

In [ ]:
# ── Cell 22: Baseline Comparison Chart ───────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
 
# All baselines from LegalSeg paper Table 3 + CLARA-RR
systems = [
    ("MTL",                     0.37, "#BFBFBF"),
    ("RhetoricLLaMA",           0.09, "#BFBFBF"),
    ("InLegalBERT(i)",          0.49, "#BFBFBF"),
    ("InLegalBERT(i-1,i)",      0.55, "#BFBFBF"),
    ("InLegalBERT(i-2,i-1,i)", 0.58, "#BFBFBF"),
    ("GNN",                     0.54, "#BFBFBF"),
    ("ToInLegalBERT",           0.62, "#BFBFBF"),
    ("Hier-BiLSTM-CRF\n(LegalSeg paper)", 0.77, "#7F7F7F"),
    (f"CLARA-RR v5\n(ours)",   test_f1, "#4472C4"),
]
 
names    = [s[0] for s in systems]
scores   = [s[1] for s in systems]
colors   = [s[2] for s in systems]
 
bars = ax.barh(names, scores, color=colors, edgecolor="white", height=0.6)
ax.axvline(x=0.77, color="#7F7F7F", linestyle="--",
           linewidth=1.5, alpha=0.7, label="BiLSTM-CRF baseline")
 
for bar, score, (name, _, color) in zip(bars, scores, systems):
    bold = "bold" if "CLARA" in name else "normal"
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{score:.4f}", va="center", fontsize=9, fontweight=bold)
 
ax.set_xlabel("Macro-F1")
ax.set_xlim(0, 1.0)
ax.set_title("CLARA-RR v5 vs. All LegalSeg Baselines (Macro-F1)",
             fontweight="bold")
 
# Highlight CLARA bar
bars[-1].set_edgecolor("#1F3864")
bars[-1].set_linewidth(2)
 
# Delta annotation
delta = test_f1 - 0.77
sign  = "+" if delta >= 0 else ""
ax.text(0.98, 0.02, f"vs. BiLSTM-CRF: {sign}{delta:.4f}",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=10, color="green" if delta >= 0 else "red",
        fontweight="bold")
 
plt.tight_layout()
SAVE("viz_05_baseline_comparison.png")
plt.show()
print("Saved: viz_05_baseline_comparison.png")

In [ ]:
# ── Cell 23: Position Distribution by Role ───────────────────
# Compute mean predicted role per position bucket (10 buckets)
N_BUCKETS = 20
bucket_role_counts = np.zeros((N_BUCKETS, CFG.NUM_LABELS))
 
# Re-evaluate test set collecting position + prediction
model.eval()
all_pos, all_pred_roles = [], []
 
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Position analysis"):
        sent_mask = batch["sent_mask"]
        positions = batch["positions"]
        input_ids       = batch["input_ids"].to(DEVICE)
        attention_masks = batch["attention_masks"].to(DEVICE)
        pos_dev         = batch["positions"].to(DEVICE)
        lex             = batch["lex"].to(DEVICE)
        sm_dev          = sent_mask.to(DEVICE)
 
        preds = model.predict(input_ids, attention_masks,
                              pos_dev, lex, sm_dev)
        for i, pred_seq in enumerate(preds):
            valid_pos = positions[i][sent_mask[i]].tolist()
            for pos, role in zip(valid_pos, pred_seq):
                bucket = min(int(pos * N_BUCKETS), N_BUCKETS - 1)
                bucket_role_counts[bucket, role] += 1
 
# Normalise to proportions per bucket
row_sums = bucket_role_counts.sum(axis=1, keepdims=True).clip(min=1)
bucket_props = bucket_role_counts / row_sums
 
fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.linspace(0, 1, N_BUCKETS)
bottom = np.zeros(N_BUCKETS)
 
role_colors = [COLORS["Facts"], COLORS["Argument"],
               COLORS["Ratio"], COLORS["Ruling_Present"]]
 
for role_id in range(CFG.NUM_LABELS):
    ax.bar(x_pos, bucket_props[:, role_id], width=1/N_BUCKETS,
           bottom=bottom, color=role_colors[role_id],
           label=CLASS_NAMES[role_id], edgecolor="white", linewidth=0.3)
    bottom += bucket_props[:, role_id]
 
ax.set_xlabel("Relative Document Position (0 = start, 1 = end)")
ax.set_ylabel("Proportion of Predictions")
ax.set_title("Predicted Role Distribution by Document Position (Test Set)",
             fontweight="bold")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
ax.set_xlim(0, 1)
 
# Add structural annotations
ax.axvline(x=0.85, color="red", linestyle="--", alpha=0.5, linewidth=1.2)
ax.text(0.86, 0.95, "RP peak\n(0.884)", color="red",
        fontsize=8, transform=ax.get_xaxis_transform(), va="top")
 
plt.tight_layout()
SAVE("viz_06_position_distribution.png")
plt.show()
print("Saved: viz_06_position_distribution.png")

In [ ]:
# ── Cell 24: Summary Dashboard ───────────────────────────────
fig = plt.figure(figsize=(14, 8))
gs  = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.4)
 
fig.suptitle("CLARA-RR v5 on LegalSeg — Results Dashboard",
             fontsize=15, fontweight="bold", y=1.01)
 
# ── Panel 1: Epoch F1 mini-curve ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs_list, val_f1s, "o-", color=COLORS["clara"], linewidth=2, markersize=6)
ax1.axhline(y=0.77, color=COLORS["baseline"], linestyle="--",
            linewidth=1.2, alpha=0.7)
ax1.scatter([best_epoch], [best_val_f1], color="green", zorder=5, s=80, marker="*")
ax1.set_title("Val Macro-F1 per Epoch")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Macro-F1")
ax1.set_ylim(0, 1.0); ax1.set_xticks(epochs_list)
ax1.text(0.05, 0.92, f"Best: {best_val_f1:.4f}",
         transform=ax1.transAxes, color="green", fontsize=9, fontweight="bold")
 
# ── Panel 2: Per-class F1 bars ──
ax2 = fig.add_subplot(gs[0, 1])
bar_colors = [COLORS[c] for c in CLASS_NAMES]
ax2.barh(CLASS_NAMES, f1, color=bar_colors, edgecolor="white")
ax2.axvline(x=test_f1, color="red", linestyle="--",
            linewidth=1.2, alpha=0.7, label=f"Macro-F1={test_f1:.3f}")
for i, v in enumerate(f1):
    ax2.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax2.set_xlim(0, 1.0)
ax2.set_title("Per-Class F1 (Test)")
ax2.legend(fontsize=8)
 
# ── Panel 3: Confusion matrix (normalised) ──
ax3 = fig.add_subplot(gs[0, 2])
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True).clip(min=1)
im = ax3.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax3.set_xticks(range(CFG.NUM_LABELS)); ax3.set_yticks(range(CFG.NUM_LABELS))
ax3.set_xticklabels([c[:4] for c in CLASS_NAMES], rotation=30, ha="right", fontsize=8)
ax3.set_yticklabels([c[:4] for c in CLASS_NAMES], fontsize=8)
ax3.set_title("Confusion Matrix\n(Recall-normalised)")
for i, j in itertools.product(range(CFG.NUM_LABELS), range(CFG.NUM_LABELS)):
    ax3.text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center",
             fontsize=8, color="white" if cm_norm[i,j] > 0.5 else "black")
plt.colorbar(im, ax=ax3, shrink=0.8)
 
# ── Panel 4: Baseline comparison mini ──
ax4 = fig.add_subplot(gs[1, 0])
baselines  = ["BiLSTM\nCRF", "ToInLegal\nBERT", "GNN", "CLARA\n(ours)"]
b_scores   = [0.77, 0.62, 0.54, test_f1]
b_colors   = [COLORS["baseline"], COLORS["baseline"],
              COLORS["baseline"], COLORS["clara"]]
bars4 = ax4.bar(baselines, b_scores, color=b_colors, edgecolor="white")
bars4[-1].set_edgecolor("#1F3864"); bars4[-1].set_linewidth(2)
ax4.axhline(y=0.77, color=COLORS["baseline"], linestyle="--",
            linewidth=1, alpha=0.6)
for bar, sc in zip(bars4, b_scores):
    ax4.text(bar.get_x() + bar.get_width()/2, sc + 0.01,
             f"{sc:.3f}", ha="center", fontsize=9,
             fontweight="bold" if sc == test_f1 else "normal")
ax4.set_ylim(0, 1.0); ax4.set_title("vs. Key Baselines")
ax4.set_ylabel("Macro-F1")
 
# ── Panel 5: Dataset class imbalance ──
ax5 = fig.add_subplot(gs[1, 1])
clara_counts_k = [c/1000 for c in clara_counts]
wedges, texts, autotexts = ax5.pie(
    clara_counts_k,
    labels=CLASS_NAMES,
    colors=[COLORS[c] for c in CLASS_NAMES],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1}
)
for at in autotexts:
    at.set_fontsize(8)
ax5.set_title("Class Distribution\n(after None-drop)")
 
# ── Panel 6: Key metrics summary ──
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis("off")
metrics_text = [
    ("Dataset",        "LegalSeg (7,120 docs)"),
    ("Classes",        "4 (after None-drop)"),
    ("Train docs",     "4,984"),
    ("Test docs",      "712"),
    ("", ""),
    ("Test Macro-F1",  f"{test_f1:.4f}"),
    ("Best Val F1",    f"{best_val_f1:.4f}"),
    ("Best Epoch",     f"{best_epoch}/{CFG.EPOCHS}"),
    ("", ""),
    ("Baseline (BiLSTM-CRF)", "0.7700"),
    ("Delta",          f"{test_f1-0.77:+.4f}"),
]
 
y_start = 0.98
for key, val in metrics_text:
    if not key:
        y_start -= 0.04; continue
    is_delta = key == "Delta"
    color = "green" if (is_delta and test_f1 >= 0.77) else \
            "red"   if is_delta else "black"
    bold  = "bold" if key in ("Test Macro-F1", "Delta") else "normal"
    ax6.text(0.02, y_start, f"{key}:", fontsize=9, fontweight="bold",
             transform=ax6.transAxes, color="gray")
    ax6.text(0.55, y_start, val, fontsize=9, fontweight=bold,
             transform=ax6.transAxes, color=color)
    y_start -= 0.08
 
ax6.set_title("Key Metrics", fontweight="bold")
ax6.add_patch(mpatches.FancyBboxPatch(
    (0, 0), 1, 1, boxstyle="round,pad=0.02",
    linewidth=1, edgecolor="#CCCCCC", facecolor="none",
    transform=ax6.transAxes
))
 
SAVE("viz_07_dashboard.png")
plt.show()
print("Saved: viz_07_dashboard.png")

In [ ]:
# ── Cell 25: Final file list ─────────────────────────────────
print("\n" + "="*60)
print("ALL VISUALISATIONS SAVED")
print("="*60)
viz_files = [
    ("viz_01_dataset_distribution.png", "Label distribution before/after remap"),
    ("viz_02_training_curves.png",       "Loss + Val F1 per epoch"),
    ("viz_03_confusion_matrix.png",      "Test confusion matrix (counts + normalised)"),
    ("viz_04_perclass_metrics.png",      "Per-class P/R/F1 bar chart"),
    ("viz_05_baseline_comparison.png",   "CLARA-RR vs all LegalSeg baselines"),
    ("viz_06_position_distribution.png", "Role distribution by document position"),
    ("viz_07_dashboard.png",             "Full results dashboard"),
]
for fname, desc in viz_files:
    path = f"{OUTPUT_DIR}/{fname}"
    exists = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"  {exists}  {fname}")
    print(f"         {desc}")
 
print(f"\nTest Macro-F1 : {test_f1:.4f}")
print(f"vs. BiLSTM-CRF: {test_f1 - 0.77:+.4f}")
print("="*60)